# Nettoyage backtest — table de recherche canonique (2014-2026)

**But** : partir de la donnée récupérée complète (corpus validé House + Sénat, tout ce qui est déclaré
— y compris obligations, munis, options, lignes sans ticker) et en dériver **une table directement
backtestable** : `data/clean/transactions_backtest_2014_2026.csv`.

- **Entonnoir A→D** — on ne retire que ce qui est *matériellement* inutilisable pour un backtest :

| Étape | Retire | Pourquoi c'est inutilisable |
|---|---|---|
| A | dates illisibles, divulgation avant transaction, année implausible | pas de chronologie fiable → impossible de dater l'entrée |
| B | sans ticker exploitable, non coté, options/obligations/munis | pas de prix de marché → rien à valoriser |
| C | opérations hors achat/vente (échanges) | pas de sens directionnel clair |
| D | montant absent | pas de taille de position |

- **Tout le doute est GARDÉ et FLAGUÉ** (dépôts tardifs, titres délistés, ETF diversifiés, lots
  multi-comptes) : aucune ligne écartée en silence — la décision revient à la recherche, pas au nettoyage.
- **Enrichissements** en sortie : parti et commissions **à la date de la transaction**, ticker prêt pour
  le join prix (renommages appliqués), flags de traçabilité, invariants garantis par assertions.
- ⚠ La table contient des **lots réels multi-comptes** (colonnes `owner` / `occurrence_index` /
  `lot_size`) : ne **jamais** appliquer `drop_duplicates()`.


In [1]:
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

# --- localisation de la racine du dépôt (dossier contenant common/ ET data/) ---
# Le code+données S1/S2 ont été regroupés sous « 00_S1S2_donnees/ ». On teste le cwd et ses parents
# (cas normal : ce notebook est ouvert depuis 00_S1S2_donnees/), puis des candidats explicites au cas où
# Jupyter démarrerait à la RACINE du dépôt — 00_S1S2_donnees/ en est alors un ENFANT, jamais atteint par
# la simple remontée vers les parents.
def _find_repo():
    here = Path.cwd()
    cands = [here, *here.parents,
             here / "00_S1S2_donnees",
             Path.home() / "Downloads" / "Jupiter" / "00_S1S2_donnees",
             Path.home() / "Downloads" / "Jupiter"]            # repli : ancienne disposition (data/ à la racine)
    for c in cands:
        if (c / "common" / "quality.py").exists() and (c / "data" / "house" / "tables").exists():
            return c
    raise RuntimeError("Racine du dépôt (contenant common/ et data/) introuvable")

REPO = _find_repo()
sys.path.insert(0, str(REPO))

# --- fonctions réutilisées (aucune logique de nettoyage réécrite ici) ---
from common.quality import load_final, _asset_bucket    # chargement + famille d'actif
from common.schema import canonical_ticker               # ticker canonique (distingue
                                                          # le fonds coté NAN de l'artefact pandas 'nan')
from common.quiver_diagnosis import _quiver_untradeable  # ticker non coté (CUSIP, $, fragment OCR…)

# --- référentiels transverses ---
REF = REPO / "data" / "reference"
renames = pd.read_csv(REF / "ticker_renames.csv")
sector_map = pd.read_csv(REF / "ticker_sector_map.csv").set_index("ticker")
print("Dépôt :", REPO)
print(f"Référentiels : {len(renames)} renommages/délistages, {len(sector_map)} tickers en carte secteur")

# --- petit utilitaire pour tracer l'entonnoir (étape, filtre, retirées, restantes) ---
funnel = []
def _step(code, label, before, after):
    funnel.append({"étape": code, "filtre": label,
                   "retirées": before - after, "restantes": after})
    print(f"[{code}] {label}\n    {before:,} → {after:,}   (retirées : {before - after:,})")

Dépôt : /Users/lemairealice/Downloads/Jupiter/00_S1S2_donnees
Référentiels : 84 renommages/délistages, 4848 tickers en carte secteur


## Chargement — le corpus complet, prêt à filtrer

`load_final` fournit le point de départ (aucun recalcul à faire ici) :
- **assemble les 4 sous-corpus** (House / Sénat × électronique / OCR) depuis les tables FINAL par année ;
- **déduplique les re-divulgations** d'une année sur l'autre (clé naturelle + rang d'occurrence — les
  lots multi-comptes réels sont préservés) ;
- **applique les corrections de lecture** documentées (dates, fourchettes, tickers résolus depuis la
  description, identités) — le figé sur disque reste intact ;
- **dérive les colonnes utiles** : `lag_days` (divulgation − transaction), `op` (buy/sell/exch),
  `amount_midpoint`, `_td` (la date de transaction en datetime), `corpus`.

In [2]:
df = load_final(REPO)
N0 = len(df)
print(f"{N0:,} transactions uniques chargées\n")
print("Répartition par sous-corpus :")
print(df["corpus"].value_counts().to_string())

print(f"\nCorrections de lecture actives (le figé sur disque reste intact) :")
print(f"  fourchettes de montant complétées : {int(df['amount_range_repaired'].sum()):,}")
print(f"  tickers résolus depuis la description de l'actif : "
      f"{int((df['ticker_source'] == 'recovered').sum()):,}")
print(f"  identité rattachée (bioguide) : {df['bioguide_id'].notna().mean():.1%} des lignes")


169,000 transactions uniques chargées

Répartition par sous-corpus :
corpus
House OCR             93261
House électronique    58728
Sénat électronique    13026
Sénat OCR              3985

Corrections de lecture actives (le figé sur disque reste intact) :
  fourchettes de montant complétées : 7,996
  tickers résolus depuis la description de l'actif : 2,540
  identité rattachée (bioguide) : 100.0% des lignes


## Montants — une seule convention

- `amount_midpoint` = **milieu exact de la fourchette STOCK Act** (les montants sont déclarés par fourchettes légales, ex. $1,001–$15,000, jamais en valeur exacte) : `($borne basse + $borne haute) / 2`,
  recalculé depuis `amount_range` pour toutes les fourchettes complètes.
- Un **montant exact** déclaré (ex. `$584`) reste tel quel.
- Le **palier ouvert** « Over $50,000,000 » est valorisé au plancher **50 M$** (pas de borne haute →
  choix conservateur).

In [3]:
# Palier ouvert « > $50M » : pas de borne haute déclarée → plancher 50 M$ appliqué uniformément.
palier_ouvert = df["amount_range"].astype(str).str.strip().eq("Over $50,000,000")
n_fix = int((palier_ouvert & (df["amount_midpoint"] != 50_000_000)).sum())
df.loc[palier_ouvert, "amount_midpoint"] = 50_000_000.0
print(f"Palier ouvert « > $50M » harmonisé au plancher : {n_fix} ligne(s).")

# Midpoint UNIFIÉ = (lo+hi)/2 exact depuis amount_range (fourchettes complètes uniquement).
import re as _re
def _mid_exact(a):
    nums = [int(x.replace(",", "")) for x in _re.findall(r"\$([\d,]+)", str(a))]
    return (nums[0] + nums[1]) / 2 if len(nums) == 2 else None
_mid = df["amount_range"].map(_mid_exact)
_maj = _mid.notna() & (df["amount_midpoint"] != _mid) & ~palier_ouvert
df.loc[_maj, "amount_midpoint"] = _mid[_maj]
print(f"Midpoint unifié (lo+hi)/2 recalculé depuis amount_range : {int(_maj.sum()):,} lignes alignées "
      f"(les sous-corpus arrondissaient différemment le demi-dollar).")

Palier ouvert « > $50M » harmonisé au plancher : 2 ligne(s).
Midpoint unifié (lo+hi)/2 recalculé depuis amount_range : 91,634 lignes alignées (les sous-corpus arrondissaient différemment le demi-dollar).


## Étape A — Dates présentes et cohérentes

Un backtest entre à la divulgation et mesure depuis la transaction : il lui faut une chronologie fiable.
On retire les lignes où :
- une des deux dates est **illisible ou absente** (`lag_days` = NaN) → impossible de savoir *quand* agir ;
- la **divulgation précède la transaction** (`lag_days` < 0) → impossible par construction (coquille du
  déposant ou lecture erronée) ;
- l'**année de transaction est implausible** (avant 2012 ou après l'année du dépôt) — garde-fou.

`lag_days` est conservé en colonne : les dépôts tardifs (> 45 j) sont **gardés et flagués** plus bas,
jamais retirés — l'information reste réelle et exploitable à sa date de publication.

In [4]:
n = len(df)
fy = pd.to_numeric(df["file_year"], errors="coerce")
mask_parse    = df["lag_days"].notna()                        # dates lisibles
mask_coherent = df["lag_days"] >= 0                           # divulgation ≥ transaction
mask_year     = (df["txn_year"] >= 2012) & (df["txn_year"] <= fy)   # année plausible
keep = mask_parse & mask_coherent & mask_year

# aperçu : quelques dates incohérentes retirées (divulgation AVANT transaction)
apercu = (df[mask_parse & (df["lag_days"] < 0)]
          [["declarant_name", "ticker", "transaction_date", "disclosure_date", "lag_days"]].head(5))
print("Exemples de dates incohérentes retirées (divulgation avant transaction) :")
print(apercu.to_string(index=False), "\n")

df = df[keep].copy()
_step("A", "dates présentes & cohérentes", n, len(df))

Exemples de dates incohérentes retirées (divulgation avant transaction) :
  declarant_name ticker transaction_date disclosure_date  lag_days
     Kevin Yoder    NaN       2014-12-17      2014-12-16      -1.0
     Kevin Yoder   SMLP       2014-12-17      2014-12-16      -1.0
Randy Neugebauer    QRE       2014-06-24      2014-06-20      -4.0
Randy Neugebauer    QRE       2014-06-24      2014-06-20      -4.0
Randy Neugebauer    NaN       2014-05-15      2014-05-14      -1.0 

[A] dates présentes & cohérentes
    169,000 → 165,748   (retirées : 3,252)


## Étape B — Actions et ETF cotés uniquement (ticker-first)

Un backtest a besoin d'un **prix**, donc d'un symbole coté. Une ligne est gardée si — et seulement si :
- son ticker se **normalise en un symbole exploitable** (`canonical_ticker` : majuscules, format classe
  d'action, sentinelles éliminées) ;
- le symbole est **réellement coté** (on écarte CUSIP, préférentielles `$`, fragments d'OCR, échéances
  obligataires) ;
- sa **famille d'actif est cotée** (on écarte options, obligations, munis, bons du Trésor).

**Ticker-first** : le ticker fait foi avant le type déclaré — une action dont le champ « type d'actif »
est vide est **gardée** grâce à son symbole (aucune perte pour un simple champ manquant).

In [5]:
n = len(df)
NON_COTE = {"bond", "muni", "gov", "option", "autre"}
mask_ticker   = df["ticker"].map(lambda t: canonical_ticker(t)[0]) != ""   # symbole non vide (le fonds coté NAN ≠ artefact pandas 'nan')
mask_tradable = ~df["ticker"].map(_quiver_untradeable)                 # réellement coté
mask_famille  = ~df["asset_type"].map(_asset_bucket).isin(NON_COTE)    # pas une famille non-cotée
keep = mask_ticker & mask_tradable & mask_famille

apercu = df[~keep][["declarant_name", "asset_description", "asset_type", "ticker"]].head(5)
print("Exemples de lignes retirées (non cotées / non tickérisées) :")
print(apercu.to_string(index=False), "\n")

avant_B = df                       # snapshot avant filtrage (réutilisé par la cellule de contrôle ci-dessous)
df = df[keep].copy()
_step("B", "actions + ETF cotés (ticker-first)", n, len(df))

Exemples de lignes retirées (non cotées / non tickérisées) :
  declarant_name                       asset_description asset_type ticker
Randy Neugebauer          QR ENERGY, LP 9.25% 08/01/2020        NaN    NaN
Randy Neugebauer          QR ENERGY, LP 9.25% 08/01/2020        NaN    NaN
Randy Neugebauer          QR ENERGY, LP 9.25% 08/01/2020        NaN    NaN
Randy Neugebauer          QR ENERGY, LP 9.25% 08/01/2020        NaN    NaN
    Lois Frankel Liberty broadband Corporation - Class C        NaN    NaN 

[B] actions + ETF cotés (ticker-first)
    165,748 → 135,977   (retirées : 29,771)


### Contrôle — pourquoi chaque ligne de l'étape B part

Une **seule cause par ligne** (dans l'ordre : ticker vide → malformé/non coté → famille non cotée), puis
un contrôle d'étanchéité : aucune famille non cotée ne doit survivre parmi les lignes gardées.

In [6]:
# On réutilise `avant_B` (état AVANT filtrage) + les 3 masques calculés à l'étape B. On ne re-filtre RIEN.
fam = avant_B["asset_type"].map(_asset_bucket)
cause = pd.Series(index=avant_B.index, dtype="object")
cause[~mask_ticker]                                = "1. ticker VIDE (aucun symbole → pas de prix)"
cause[mask_ticker & ~mask_tradable]                = "2. ticker MALFORMÉ ($, espace, fragment OCR)"
cause[mask_ticker & mask_tradable & ~mask_famille] = "3. OPTION / OBLIGATION (famille non-cotée)"
ret = cause.dropna()

print(f"Pourquoi les {len(ret):,} lignes de l'étape B partent (une seule cause par ligne) :")
print(ret.value_counts().sort_index().to_string())
print()

print(f"Contrôle d'étanchéité — familles des {int(keep.sum()):,} GARDÉES (equity/ETF seulement) :")
print(fam[keep].value_counts().to_string())
fuite = int(fam[keep].isin(list(NON_COTE)).sum())
print(f"  → non-cotées ayant fui dans les gardées : {fuite}   (0 = filtre étanche)")
print()

rescue = int((keep & (fam == "manquant")).sum())
print(f"Rescue « ticker-first » : {rescue:,} actions/ETF à asset_type vide GARDÉES grâce au ticker.")

Pourquoi les 29,771 lignes de l'étape B partent (une seule cause par ligne) :
1. ticker VIDE (aucun symbole → pas de prix)    27006
2. ticker MALFORMÉ ($, espace, fragment OCR)      164
3. OPTION / OBLIGATION (famille non-cotée)       2601

Contrôle d'étanchéité — familles des 135,977 GARDÉES (equity/ETF seulement) :
asset_type
action      112326
manquant     21878
fonds         1773
  → non-cotées ayant fui dans les gardées : 0   (0 = filtre étanche)

Rescue « ticker-first » : 21,878 actions/ETF à asset_type vide GARDÉES grâce au ticker.


## Étape C — Direction claire (achat / vente)

- On garde **achats et ventes** (les deux sens : le backtest décide — long, short, étude d'événement).
- On retire les **échanges** (`exchange`) et rares `other` : pas d'économie directionnelle claire.

In [7]:
n = len(df)
df = df[df["op"].isin(["buy", "sell"])].copy()
_step("C", "direction ∈ {achat, vente}", n, len(df))

[C] direction ∈ {achat, vente}
    135,977 → 135,151   (retirées : 826)


## Étape D — Montant présent

- Dimensionner une position exige un notionnel : on retire les lignes sans `amount_midpoint` (< 1 % —
  fourchette absente du formulaire).

In [8]:
n = len(df)
df = df[df["amount_midpoint"].notna()].copy()
_step("D", "montant présent (amount_midpoint)", n, len(df))

[D] montant présent (amount_midpoint)
    135,151 → 134,464   (retirées : 687)


## Enrichissements — de l'information EN PLUS, aucune ligne retirée

1. **Parti à la date de la transaction** — les changements de parti en cours de mandat sont datés
   (`party_affiliations` du référentiel officiel des législateurs).
2. **Commissions du Congrès de la transaction** — snapshots par Congrès 113-119
   (`data/reference/committees_snapshots/`), bascule au **3 janvier** des années impaires ;
   `committees_key_flag` = fiscalité / défense / renseignement / banque (la liste complète des
   commissions est exportée : la recherche peut redéfinir son propre flag).
3. **Ticker prêt pour le join prix** — `ticker` reste FIDÈLE à la déclaration ; `ticker_yahoo` = symbole
   canonique (format Yahoo) avec **renommages et fusions appliqués** (`data/reference/ticker_renames.csv`) ;
   les titres **délistés sont typés** (rachat, faillite) au lieu de disparaître en silence.
4. **Classe d'actif et secteur** — carte transverse `data/reference/ticker_sector_map.csv` : les actions
   ont un secteur GICS et un ETF SPDR proxy ; les **ETF diversifiés sont gardés** avec `is_broad_etf`
   (pas de secteur GICS par nature).
5. **Flags de traçabilité** — dépôts tardifs (> 45 j / > 365 j), `lot_size` (lignes identiques d'un même
   lot multi-comptes = transactions réelles).

In [9]:
# 1) Parti point-in-time — depuis les YAML congress-legislators versionnés (offline),
#    en éclatant party_affiliations (sous-périodes de switch EN COURS de mandat).
import yaml
try:
    from yaml import CSafeLoader as _YL
except ImportError:
    from yaml import SafeLoader as _YL

_ppl = []
for f in ("legislators-current.yaml", "legislators-historical.yaml"):
    _ppl += yaml.load((REPO / "data" / "house" / "reference" / f).read_text(), Loader=_YL)

PARTY_SPANS = {}
for p in _ppl:
    bio = (p.get("id") or {}).get("bioguide")
    if not bio:
        continue
    spans = []
    for t in (p.get("terms") or []):
        st, en = t.get("start"), t.get("end") or "2100-01-01"
        if not st:
            continue
        affs = t.get("party_affiliations")
        if affs:
            for a in affs:
                spans.append((pd.Timestamp(a.get("start") or st),
                              pd.Timestamp(a.get("end") or en), a.get("party")))
        else:
            spans.append((pd.Timestamp(st), pd.Timestamp(en), t.get("party")))
    if spans:
        PARTY_SPANS[bio] = sorted(spans)

def party_at(bio, d):
    spans = PARTY_SPANS.get(bio)
    if not spans or pd.isna(d):
        return None
    for st, en, pty in spans:
        if st <= d <= en:
            return pty
    before = [s for s in spans if s[0] <= d]
    return (before[-1] if before else spans[0])[2]

_old = df["party"].copy()
df["party"] = [party_at(b, d) or old for b, d, old in zip(df["bioguide_id"], df["_td"], _old)]
_chg = int((df["party"] != _old).sum())
print(f"Parti à la date du trade : {_chg} ligne(s) où il diffère du parti du dernier mandat (switchs en cours de mandat)")
print(df.loc[df["party"] != _old, ["declarant_name", "transaction_date", "party"]]
        .assign(ancien=_old[df["party"] != _old]).head(8).to_string(index=False))

Parti à la date du trade : 24 ligne(s) où il diffère du parti du dernier mandat (switchs en cours de mandat)
     declarant_name transaction_date      party      ancien
       Justin Amash       2015-11-10 Republican Libertarian
Joseph Manchin, III       2017-04-21   Democrat Independent
      Paul Mitchell       2018-12-17 Republican Independent
      Paul Mitchell       2018-12-17 Republican Independent
      Paul Mitchell       2018-12-12 Republican Independent
      Paul Mitchell       2018-12-12 Republican Independent
       Justin Amash       2018-11-01 Republican Libertarian
       Justin Amash       2018-11-01 Republican Libertarian


In [10]:
# 2) Commissions point-in-time — snapshots par Congrès (data/reference/committees_snapshots).
from collections import defaultdict

def congress_of(d):
    # Un Congrès commence le 3 JANVIER des années impaires : les trades des 1-2 janvier d'une année
    # impaire appartiennent encore au Congrès sortant.
    if pd.isna(d):
        return None
    y = d.year
    start_congress_year = y if y % 2 == 1 else y - 1
    if y % 2 == 1 and (d.month, d.day) < (1, 3):
        start_congress_year -= 2
    return 113 + (start_congress_year - 2013) // 2

SNAPS = {}
for cg in range(113, 120):
    d = REPO / "data" / "reference" / "committees_snapshots" / str(cg)
    mem = yaml.load((d / "membership.yaml").read_text(), Loader=_YL)
    com = yaml.load((d / "committees.yaml").read_text(), Loader=_YL)
    code_to_name = {c["thomas_id"]: c["name"] for c in com if "thomas_id" in c}
    bio2c = defaultdict(set)
    for code_, members in mem.items():
        cname = code_to_name.get(code_, code_)
        for m in members:
            if m.get("bioguide"):
                bio2c[m["bioguide"]].add(cname)
    SNAPS[cg] = {b: "; ".join(sorted(cs)) for b, cs in bio2c.items()}
    print(f"  Congrès {cg} : {len(SNAPS[cg])} membres avec commissions")

# Commissions « clés » : patterns LARGES documentés (fiscalité + défense + renseignement + banque,
# les deux chambres). La liste COMPLÈTE des commissions est exportée : la recherche peut redéfinir
# son propre flag sans re-générer la table.
KEY_PATTERNS = ("Financial Services", "Committee on Finance", "Ways and Means",
                "Banking", "Armed Services", "Intelligence")

df["congress"] = [congress_of(d) for d in df["_td"]]
_snap_get = lambda b, cg: (SNAPS.get(cg) or {}).get(b) if pd.notna(cg) and cg in SNAPS else None
df["committee_membership"] = [_snap_get(b, cg) for b, cg in zip(df["bioguide_id"], df["congress"])]
df["committees_key_flag"] = [any(p in m for p in KEY_PATTERNS) if isinstance(m, str) else pd.NA
                             for m in df["committee_membership"]]
print(f"\ncommissions PIT résolues : {df['committee_membership'].notna().mean():.1%} des lignes "
      f"| flag clé : {df['committees_key_flag'].mean():.1%}")

  Congrès 113 : 531 membres avec commissions
  Congrès 114 : 536 membres avec commissions


  Congrès 115 : 528 membres avec commissions
  Congrès 116 : 529 membres avec commissions


  Congrès 117 : 526 membres avec commissions
  Congrès 118 : 529 membres avec commissions


  Congrès 119 : 528 membres avec commissions

commissions PIT résolues : 99.1% des lignes | flag clé : 54.6%


In [11]:
# 3) Ticker canonique + renommages/délistages — `ticker` reste FIDÈLE à la déclaration.
_canon = df["ticker"].map(canonical_ticker)
df["ticker_yahoo"] = [c[0] for c in _canon]
df["flag_ticker"] = [c[1] for c in _canon]

_ren = renames.set_index("ticker_ancien")
_map_new = _ren.loc[(_ren["ticker_nouveau"].notna()) & (_ren["ticker_nouveau"] != ""), "ticker_nouveau"]
_n_ren = int(df["ticker_yahoo"].isin(_map_new.index).sum())
df["ticker_yahoo"] = df["ticker_yahoo"].map(lambda t: _map_new.get(t, t))

df["delist_type"] = df["ticker_yahoo"].map(_ren["type"]).where(
    df["ticker_yahoo"].isin(_ren.index[_ren["ticker_nouveau"].isna() | (_ren["ticker_nouveau"] == "")]))
# jambe absorbée d'une fusion : prix du successeur INVALIDE avant la fusion → prudence au join prix
_caution = set(_ren.index[(_ren.get("historique_valide") == "post_fusion_seulement")]) | \
           set(_ren.index[_ren["type"] == "recyclage_attention"])
_orig_canon = pd.Series([c[0] for c in _canon], index=df.index)
df["flag_price_caution"] = _orig_canon.isin(_caution) | df["ticker_yahoo"].isin(_caution)
df["is_delisted"] = df["delist_type"].notna()

print(f"ticker_yahoo : {_n_ren:,} lignes re-symbolisées (renommages/fusions), "
      f"{int(df['is_delisted'].sum()):,} lignes sur titres délistés (type le plus fréquent : "
      f"{df['delist_type'].mode().iat[0] if df['delist_type'].notna().any() else '—'}), "
      f"{int(df['flag_price_caution'].sum()):,} lignes « prudence join prix » (recyclage / jambe absorbée)")
print("flag_ticker :", df["flag_ticker"].value_counts().to_dict())

ticker_yahoo : 2,817 lignes re-symbolisées (renommages/fusions), 3,542 lignes sur titres délistés (type le plus fréquent : rachat_delisting), 505 lignes « prudence join prix » (recyclage / jambe absorbée)
flag_ticker : {'ok': 133605, 'classe_convertie': 836, 'contient_chiffre': 23}


In [12]:
# 4) Classe d'actif / secteur GICS / ETF proxy — carte transverse corrigée (les ETF diversifiés
#    n'ont pas de secteur GICS par nature — leur en donner un serait faux).
_key = pd.Series([c[0] for c in _canon], index=df.index)          # symbole déclaré canonisé (pré-rename)
df["asset_class"] = _key.map(sector_map["asset_class"]).fillna("unknown")
_sec_new = _key.map(sector_map["sector_gics"])
_etf_new = _key.map(sector_map["etf_proxy"])
# priorité à la carte quand elle connaît le ticker ; sinon on GARDE la colonne des FINAL (jamais de trou créé)
_known = _key.isin(sector_map.index)
df["sector_gics"] = _sec_new.where(_known & _sec_new.notna() & (_sec_new != ""), df["sector_gics"])
df.loc[_known & df["asset_class"].isin(["etf_broad", "etf_sector"]), "sector_gics"] = pd.NA
df["etf_proxy"] = _etf_new.where(_known & _etf_new.notna() & (_etf_new != ""), df["etf_proxy"])
df["is_broad_etf"] = df["asset_class"] == "etf_broad"

_stk = df["asset_class"].eq("stock")
print(f"asset_class : {df['asset_class'].value_counts().to_dict()}")
print(f"secteur GICS rempli (actions) : {df.loc[_stk, 'sector_gics'].notna().mean():.1%} "
      f"| etf_proxy rempli : {df['etf_proxy'].notna().mean():.1%}")

asset_class : {'stock': 128974, 'unknown': 3300, 'etf_broad': 2022, 'etf_sector': 168}
secteur GICS rempli (actions) : 100.0% | etf_proxy rempli : 98.8%


In [13]:
# 5) Flags de traçabilité — dépôts tardifs (l'info reste RÉELLE et exploitable à disclosure_date,
#    on FLAGUE, on ne retire pas) + lots multi-comptes.
df["flag_late_filing"] = df["lag_days"] > 45
df["flag_very_late_filing"] = df["lag_days"] > 365
df["lot_size"] = df.groupby("natural_key_hash")["natural_key_hash"].transform("size")
print(f"dépôts > 45 j : {int(df['flag_late_filing'].sum()):,} ({df['flag_late_filing'].mean():.1%}) "
      f"| > 365 j : {int(df['flag_very_late_filing'].sum()):,}")
print(f"lignes appartenant à un lot multi-lignes (mêmes 7 champs de clé) : "
      f"{int((df['lot_size'] > 1).sum()):,} — comptes multiples réels (owner/occurrence), PAS des doublons")

dépôts > 45 j : 16,349 (12.2%) | > 365 j : 3,075
lignes appartenant à un lot multi-lignes (mêmes 7 champs de clé) : 11,873 — comptes multiples réels (owner/occurrence), PAS des doublons


## Récapitulatif de l'entonnoir

In [14]:
rows = [{"étape": "—", "filtre": "départ (load_final)", "retirées": 0, "restantes": N0}] + funnel
recap = pd.DataFrame(rows)
print(recap.to_string(index=False))
print(f"\nDonnée propre finale : {len(df):,} lignes  ({100 * len(df) / N0:.1f} % du panel de départ)\n")

print("Par chambre :");     print(df["chamber"].value_counts().to_string())
print("\nPar sous-corpus :"); print(df["corpus"].value_counts().to_string())
print("\nPar sens :");        print(df["op"].value_counts().to_string())

étape                             filtre  retirées  restantes
    —                départ (load_final)         0     169000
    A       dates présentes & cohérentes      3252     165748
    B actions + ETF cotés (ticker-first)     29771     135977
    C         direction ∈ {achat, vente}       826     135151
    D  montant présent (amount_midpoint)       687     134464

Donnée propre finale : 134,464 lignes  (79.6 % du panel de départ)

Par chambre :
chamber
house     122814
senate     11650

Par sous-corpus :
corpus
House OCR             75205
House électronique    47609
Sénat électronique     9729
Sénat OCR              1921

Par sens :
op
buy     68250
sell    66214


## Export — `data/clean/transactions_backtest_2014_2026.csv` (table de recherche canonique)

| Colonne | Sens |
|---|---|
| `bioguide_id`, `member_name`, `chamber`, `state_district` | identité du déposant (bioguide TOUJOURS rempli) |
| `party` | parti **à la date de la transaction** (les switchs en cours de mandat sont datés) |
| `committee_membership`, `committees_key_flag`, `congress` | commissions **du Congrès de la transaction** (snapshots 113-119) ; flag = fiscalité/défense/renseignement/banque |
| `owner`, `occurrence_index`, `lot_size` | compte (Self/Spouse/Joint/Child), n° d'occurrence dans le dépôt, taille du lot — **ne jamais `drop_duplicates()`** |
| `ticker` | symbole FIDÈLE à la déclaration |
| `ticker_yahoo`, `flag_ticker` | symbole canonique pour le join prix (format Yahoo + renommages appliqués) et statut de normalisation |
| `is_delisted`, `delist_type`, `flag_price_caution` | délistage typé (rachat/faillite) et prudence prix (symbole recyclé, jambe absorbée de fusion) |
| `asset_class`, `asset_type`, `is_broad_etf` | stock / etf_sector / etf_broad / unknown ; type déclaré ; ETF diversifié |
| `sector_gics`, `etf_proxy` | secteur GICS (actions : 100 % rempli) et SPDR proxy — un ETF diversifié n'a pas de secteur |
| `direction` | buy / sell |
| `amount_midpoint`, `amount_range`, `amount_range_repaired` | milieu EXACT de la fourchette (convention unique), fourchette source, et booléen = fourchette complétée à la lecture (borne manquante reconstituée) |
| `transaction_date`, `disclosure_date`, `lag_days`, `flag_late_filing`, `flag_very_late_filing` | dates + retard de dépôt (> 45 j légal, > 365 j) |
| `doc_id`, `provenance`, `ticker_source`, `natural_key_hash` | traçabilité document / pipeline / résolution |
| `asset_description` | libellé source de l'actif |

**Invariants vérifiés par assertion à chaque export** : bioguide, ticker, montant, direction ∈
{buy, sell}, chronologie (divulgation ≥ transaction) et clé naturelle remplis sur **100 % des lignes**.

In [15]:
OUT_DIR = REPO / "data" / "clean"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT = OUT_DIR / "transactions_backtest_2014_2026.csv"

COLS = ["bioguide_id", "member_name", "party", "chamber", "state_district",
        "committee_membership", "committees_key_flag", "congress",
        "owner", "occurrence_index", "lot_size",
        "ticker", "ticker_yahoo", "flag_ticker", "is_delisted", "delist_type", "flag_price_caution",
        "asset_class", "asset_type", "is_broad_etf", "sector_gics", "etf_proxy",
        "direction", "amount_midpoint", "amount_range", "amount_range_repaired",
        "transaction_date", "disclosure_date", "lag_days", "flag_late_filing", "flag_very_late_filing",
        "doc_id", "provenance", "ticker_source", "natural_key_hash", "asset_description"]

export = (df.rename(columns={"declarant_name": "member_name", "op": "direction"})
            .reindex(columns=COLS))

# Contrôles finaux — la table est CERTIFIÉE sur ces invariants :
assert export["bioguide_id"].notna().all() and (export["bioguide_id"] != "").all(), "bioguide manquant"
assert export["ticker"].notna().all(), "ticker manquant"
assert export["amount_midpoint"].notna().all(), "montant manquant"
assert export["direction"].isin(["buy", "sell"]).all(), "direction invalide"
assert (export["lag_days"] >= 0).all(), "chronologie incohérente"
assert export["natural_key_hash"].notna().all(), "hash manquant"

export.to_csv(OUT, index=False)
print(f"Écrit : {OUT}")
print(f"{len(export):,} lignes × {export.shape[1]} colonnes — tous les invariants vérifiés")
print(f"\nRépartition par chambre × ère :")
_y = pd.to_datetime(export["transaction_date"], errors="coerce").dt.year
print(pd.crosstab(export["chamber"], _y < 2020).rename(columns={True: "2014-2019", False: "2020-2026"}).to_string())

Écrit : /Users/lemairealice/Downloads/Jupiter/00_S1S2_donnees/data/clean/transactions_backtest_2014_2026.csv
134,464 lignes × 36 colonnes — tous les invariants vérifiés

Répartition par chambre × ère :
transaction_date  2020-2026  2014-2019
chamber                               
house                 66108      56706
senate                 4539       7111
